# VCC 2026 data explorer

In [ ]:
import sys, json, math, time, warnings
from pathlib import Path
import numpy as np, pandas as pd, h5py, anndata as ad, scipy.sparse as sp
import matplotlib.pyplot as plt
from scipy.special import expit
from IPython.display import display, Markdown
try:
    from anndata.io import read_elem
except ImportError:
    from anndata.experimental import read_elem
warnings.filterwarnings('ignore', category=RuntimeWarning)

V = Path.home() / 'vcc'
DATA, LIB = V / 'data', V / 'lib'
WORK = LIB / 'review_v4'                      # the pipeline's working folder (read-only here)
CACHE = LIB / 'explore'; CACHE.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(V / 'code'))            # reuse the pipeline's own helpers
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40); pd.set_option('display.precision', 4)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})

GENES = pd.read_csv(DATA / 'gene_names.csv')['gene_name'].astype(str).to_numpy()
GENE_POS = {g: i for i, g in enumerate(GENES)}
PANEL = pd.read_csv(DATA / 'pert_counts.csv')['target_gene'].astype(str).to_numpy()
CONTROL = 'non-targeting'
PC = 0.5                                       # CPM pseudocount for exploratory log fold changes
MIN_CELLS = 20                                 # same minimum the pipeline uses; smaller rows are mostly noise

# key -> (file, storage kind, biological background)
SOURCES = {
    'HCT116':     (LIB / 'orion/HCT116_pseudobulk.h5ad',      'sums',    'HCT116'),
    'HEK293T':    (LIB / 'orion/HEK293T_pseudobulk.h5ad',     'sums',    'HEK293T'),
    'CD4_Stim48': (LIB / 'tcell/TCELL_Stim48hr_effects.h5ad', 'effects', 'CD4_T'),
    'CD4_Rest':   (LIB / 'tcell/TCELL_Rest_effects.h5ad',     'effects', 'CD4_T'),
    'H1':         (LIB / 'h1/H1_full_2025.h5ad',              'cells',   'H1'),
}
LINES = {k: (DATA / f'context_{k}.h5ad', 'cells', f'line {k}') for k in 'ABC'}
ALL = {**SOURCES, **LINES}
missing = [str(p) for p, _, _ in ALL.values() if not p.exists()]
print(f'{len(GENES):,} genes | {len(PANEL)} challenge targets | cache: {CACHE}')
print('missing files:', missing or 'none')

## 1. Header and footer of every file

Each `.h5ad` holds a cell (or row) table `obs`, a gene table `var`, a matrix `X`, and optionally extra matrices in `layers`. Files are opened in **backed** mode, so only the rows shown are read from disk.

What each file stores:

| Context | Rows are | `X` / layers |
|---|---|---|
| HCT116, HEK293T | one row per knockdown, plus one aggregated control row | layers `sum_counts`, `sum_cpm`, `sum_cpm2`, `n_expr`: per-gene sums over that row's cells |
| CD4_T Stim48hr / Rest | one row per knockdown | `X` = log2 fold change vs control; layer `se`; control CPM in `var['ctrl_cpm']` |
| H1 | one row per **cell** | `X` = raw UMI counts |
| A, B, C | one row per **control cell** | `X` = raw UMI counts |

In [ ]:
def _block(matrix, rows, cols, gene_names):
    m = matrix[rows[0]:rows[1]]
    m = m.toarray() if sp.issparse(m) else np.asarray(m)
    return pd.DataFrame(m[:, cols[0]:cols[1]], columns=gene_names[cols[0]:cols[1]])

def peek(name, n=5, k=8):
    path, kind, background = ALL[name]
    display(Markdown(f'### {name}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB  ·  background: {background}'))
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        x_kind = type(a.X).__name__
        print(f'shape {a.n_obs:,} rows x {a.n_vars:,} genes | X: {x_kind} | layers: {[k for k in a.layers.keys() if k]} | uns: {list(a.uns.keys())}')
        print(f'genes also in the 2026 list: {np.isin(genes, GENES).sum():,} / {len(GENES):,}')
        display(Markdown('**obs (row table): first and last rows**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var (gene table): first and last genes**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        mats = {'X': a.X} if kind != 'sums' else {'layers/sum_counts': a.layers['sum_counts'], 'layers/n_expr': a.layers['n_expr']}
        if kind == 'filtered': mats['layers/se'] = a.layers['se']      # log2FC and its standard error
        for label, m in mats.items():
            display(Markdown(f'**{label}: first {n} rows x first {k} genes, then last {n} rows x last {k} genes**'))
            head = _block(m, (0, n), (0, k), genes); head.index = a.obs_names[:n]
            tail = _block(m, (a.n_obs - n, a.n_obs), (a.n_vars - k, a.n_vars), genes); tail.index = a.obs_names[-n:]
            display(head); display(tail)
        for key, value in a.uns.items():
            print(f'uns[{key!r}]: {str(value)[:300]}')
    finally:
        a.file.close()

display(Markdown('### gene_names.csv and pert_counts.csv'))
display(pd.concat([pd.read_csv(DATA/'gene_names.csv').head(3), pd.read_csv(DATA/'gene_names.csv').tail(3)]))
display(pd.concat([pd.read_csv(DATA/'pert_counts.csv').head(3), pd.read_csv(DATA/'pert_counts.csv').tail(3)]))

### Filtered datasets (`lib/filters/`)

One file per training dataset, built by `scripts/knockdown_filters.py`. Rows are the knockdowns that passed all three filters (at least 100 cells, target expressed in that dataset's controls, target dropped by at least 50% with adjusted p < 0.05). Columns are the challenge genes the dataset measured. `X` is the log2 fold change vs that dataset's controls. `obs` holds each knockdown's QC numbers and `var` each gene's control expression.

In [ ]:
# Self-contained: does not depend on peek(), so it always shows both the log2FC and the SE tables.
def peek_filtered(path, n=5, k=8):
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        display(Markdown(f'### {path.stem}  ·  `{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB'))
        print(f'{a.n_obs:,} kept knockdowns x {a.n_vars:,} genes | layers: {list(a.layers.keys())}')
        display(Markdown('**obs: one row per knockdown (first and last)**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var: one row per gene, control expression (first and last)**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        for label, m in (('X = log2 fold change vs controls', a.X), ('layers/se = standard error of that log2FC', a.layers['se'])):
            display(Markdown(f'**{label}: first {n} knockdowns x first {k} genes, then last {n} x last {k}**'))
            head = pd.DataFrame(np.asarray(m[:n, :k]), index=a.obs_names[:n], columns=genes[:k])
            tail = pd.DataFrame(np.asarray(m[a.n_obs - n:, a.n_vars - k:]), index=a.obs_names[-n:], columns=genes[-k:])
            display(head); display(tail)
    finally:
        a.file.close()

FILTERED = sorted((LIB / 'filters').glob('*.h5ad'))
print('filtered files:', [p.name for p in FILTERED] or 'none: run scripts/knockdown_filters.py first')
for path in FILTERED:
    peek_filtered(path)


### Panel tables (`lib/sources/`)

One table per source, built by `scripts/build_panel_sources.py`. Every table has the same shape: 575 panel targets (our 300, then H1's other targets) × the 18,533 genes in the 2026 order. It has two layers, `lfc` and `se`, and `X` is empty. A whole row is NaN when the knockdown failed filters 1–3 or wasn't screened (`obs.status` says which). A whole column is NaN when the gene isn't measured or is below 1 CPM in that source's controls. `coverage.csv` lists, per target, which sources kept it.

In [ ]:
# Self-contained: prints head and tail of obs, var, lfc and se for every panel table, then the coverage table.
def peek_panel(path, n=5, k=8):
    a = ad.read_h5ad(path, backed='r')
    try:
        genes = a.var_names.astype(str).to_numpy()
        kept = int((a.obs['status'] == 'kept').sum()); measurable = int(a.var['ctrl_cpm'].ge(1).sum())
        display(Markdown(f'### {path.parent.name}  ·  `{path.parent.name}/{path.name}`  ·  {path.stat().st_size/1e9:.2f} GB'))
        print(f'{a.n_obs:,} targets x {a.n_vars:,} genes | layers: {list(a.layers.keys())} | '
              f'{kept} targets kept, {measurable:,} measurable genes')
        display(Markdown('**obs: one row per panel target (first and last)**')); display(pd.concat([a.obs.head(n), a.obs.tail(n)]))
        display(Markdown('**var: one row per gene, control expression (first and last)**')); display(pd.concat([a.var.head(n), a.var.tail(n)]))
        for key, label in (('lfc', 'log2 fold change vs controls'), ('se', 'standard error of that log2FC')):
            m = a.layers[key]
            display(Markdown(f'**layers/{key} = {label}: first {n} targets x first {k} genes, then last {n} x last {k}**'))
            display(pd.DataFrame(np.asarray(m[:n, :k]), index=a.obs_names[:n], columns=genes[:k]))
            display(pd.DataFrame(np.asarray(m[a.n_obs - n:, a.n_vars - k:]), index=a.obs_names[-n:], columns=genes[-k:]))
    finally:
        a.file.close()

PANELS = sorted((LIB / 'sources').glob('*/effects_panel.h5ad'))
print('panel files:', [str(p.relative_to(LIB)) for p in PANELS] or 'none: run scripts/build_panel_sources.py first')
for path in PANELS:
    peek_panel(path)

COVERAGE = pd.read_csv(LIB / 'sources/coverage.csv', index_col='target')
display(Markdown('### `coverage.csv`: which sources kept each target (first and last)'))
display(pd.concat([COVERAGE.head(), COVERAGE.tail()]))


## 2. Pseudobulk every context, then look at noise and spread

**Pseudobulk** = one expression profile per group of cells. For every context we build a matrix with one row per group (row 0 is always the control population, then one row per knockdown) and one column per gene in the 18,533-gene challenge list. Genes a context did not measure are `NaN`.

Per row and gene we keep:

* `mean_cpm`: mean over the group's cells of per-cell CPM, $\bar y_{pg} = \frac{1}{n_p}\sum_{c\in p} 10^6\,x_{cg}/L_c$. This is the evaluator's DE convention.
* `pooled_cpm`: ratio of sums, $10^6\sum_{c\in p}x_{cg} / \sum_{c\in p}L_c$. This is what the transfer model uses as a baseline.
* `detection`: fraction of the group's cells with at least one count.

How each kind of file becomes a pseudobulk:

| Kind | Source of each row |
|---|---|
| sums (Orion) | `sum_cpm / n_cells`, `sum_counts / row total`, `n_expr / n_cells`. Exact. |
| cells (H1, A, B, C) | streamed from raw counts, 4,096 cells at a time. Exact. |
| effects (T cells) | **reconstructed**: $(\text{ctrl\_cpm}+0.5)\,2^{\text{LFC}}-0.5$, floored at 0, because no cells were released. No detection. |

Results are saved to `~/vcc/lib/explore/pseudobulk/<context>/` as `.npy` arrays you can memory-map later.

In [ ]:
PB = CACHE / 'pseudobulk'

def _gene_map(names):
    pos = {g: i for i, g in enumerate(np.asarray(names, dtype=str))}
    return np.array([pos.get(g, -1) for g in GENES], dtype=np.int64)

def _aligned(block, src):
    out = np.full((block.shape[0], len(GENES)), np.nan, dtype=np.float32)
    ok = src >= 0; out[:, ok] = block[:, src[ok]]; return out

def _save_meta(out, **meta):
    (out / 'meta.json').write_text(json.dumps(meta, indent=1, default=float))

def build_sums(key, path, chunk=512):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        names = obs.index.astype(str).to_numpy(); n = obs['n_cells'].to_numpy(float)
        src = _gene_map(var.index.astype(str)); ok = src >= 0
        measured = np.zeros(len(GENES), bool)
        measured[ok] = var['measured'].to_numpy(bool)[src[ok]] if 'measured' in var else True
        ctrl = int(np.flatnonzero(names == CONTROL)[0])
        out_row = np.empty(len(names), int); out_row[ctrl] = 0
        out_row[np.arange(len(names)) != ctrl] = np.arange(1, len(names))
        arrays = {k: np.lib.format.open_memmap(out / f'{k}.npy', 'w+', np.float32, (len(names), len(GENES)))
                  for k in ('mean_cpm', 'pooled_cpm', 'detection')}
        umi = np.zeros(len(names)); ngenes = np.zeros(len(names)); ctrl_var = None
        L = {k: f[f'layers/{k}'] for k in ('sum_counts', 'sum_cpm', 'sum_cpm2', 'n_expr')}
        for b in range(0, len(names), chunk):
            sl = slice(b, min(b + chunk, len(names))); nn = n[sl][:, None]
            cnt = L['sum_counts'][sl].astype(np.float64); tot = cnt.sum(1, keepdims=True)
            mean = L['sum_cpm'][sl] / nn; det = L['n_expr'][sl] / nn
            rows = out_row[sl]
            for k, v in (('mean_cpm', mean), ('pooled_cpm', cnt / np.maximum(tot, 1) * 1e6), ('detection', det)):
                al = _aligned(v, src); al[:, ~measured] = np.nan; arrays[k][rows] = al
            umi[rows] = tot[:, 0] / n[sl]; ngenes[rows] = (cnt > 0).sum(1)
            if sl.start <= ctrl < sl.stop:
                i = ctrl - sl.start
                m2 = L['sum_cpm2'][ctrl] / n[ctrl]
                ctrl_var = _aligned((m2 - mean[i] ** 2)[None, :], src)[0]
            print(f'  {key}: {sl.stop:,}/{len(names):,}', end='\r')
        for a_ in arrays.values(): a_.flush()
    targets = np.empty(len(names), dtype=object); targets[out_row] = names
    np.save(out / 'targets.npy', targets.astype(str)); np.save(out / 'n_cells.npy', n[np.argsort(out_row)])
    np.save(out / 'umi_per_cell.npy', umi); np.save(out / 'genes_detected.npy', ngenes)
    np.save(out / 'ctrl_var_cpm.npy', np.maximum(ctrl_var, 0)); np.save(out / 'measured.npy', measured)
    _save_meta(out, kind='sums', source=str(path), exact=True)
    (out / 'done').touch(); print(f'\n  {key}: wrote {out}')

def build_cells(key, path, obs_col='target_gene', block=4096):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    a = ad.read_h5ad(path, backed='r')
    try:
        labels = a.obs[obs_col].astype(str).to_numpy() if obs_col in a.obs else np.full(a.n_obs, CONTROL)
        groups = np.array([CONTROL] + sorted(set(labels) - {CONTROL}))
        code = pd.Index(groups).get_indexer(labels); k, G = len(groups), a.n_vars
        n = np.bincount(code, minlength=k).astype(float)
        s_cnt, s_cpm, s_cpm2, s_det = (np.zeros((k, G)) for _ in range(4))
        lib = np.zeros(a.n_obs); ngen = np.zeros(a.n_obs, np.int32)
        for start in range(0, a.n_obs, block):
            m = a.X[start:start + block]
            m = (m.tocsr() if sp.issparse(m) else sp.csr_matrix(np.asarray(m))).astype(np.float64)
            l = np.asarray(m.sum(1)).ravel(); stop = start + len(l)
            lib[start:stop] = l; ngen[start:stop] = np.diff(m.indptr)
            ind = sp.csr_matrix((np.ones(len(l)), (code[start:stop], np.arange(len(l)))), shape=(k, len(l)))
            cpm = sp.diags(1e6 / np.maximum(l, 1)) @ m
            s_cnt += (ind @ m).toarray(); s_cpm += (ind @ cpm).toarray()
            s_cpm2 += (ind @ cpm.multiply(cpm)).toarray(); s_det += (ind @ (m > 0).astype(np.float64)).toarray()
            print(f'  {key}: {stop:,}/{a.n_obs:,} cells', end='\r')
        src = _gene_map(a.var_names.astype(str))
    finally:
        a.file.close()
    mean = s_cpm / n[:, None]
    np.save(out / 'mean_cpm.npy', _aligned(mean, src))
    np.save(out / 'pooled_cpm.npy', _aligned(s_cnt / np.maximum(s_cnt.sum(1, keepdims=True), 1) * 1e6, src))
    np.save(out / 'detection.npy', _aligned(s_det / n[:, None], src))
    np.save(out / 'var_cpm.npy', _aligned(np.maximum(s_cpm2 / n[:, None] - mean ** 2, 0), src))
    np.save(out / 'ctrl_var_cpm.npy', _aligned(np.maximum(s_cpm2[:1] / n[0] - mean[:1] ** 2, 0), src)[0])
    np.save(out / 'targets.npy', groups.astype(str)); np.save(out / 'n_cells.npy', n)
    np.save(out / 'umi_per_cell.npy', np.bincount(code, weights=lib, minlength=k) / n)
    np.save(out / 'genes_detected.npy', (s_cnt > 0).sum(1))
    np.save(out / 'cell_library.npy', lib); np.save(out / 'cell_genes.npy', ngen); np.save(out / 'cell_group.npy', code.astype(np.int32))
    np.save(out / 'measured.npy', src >= 0)
    _save_meta(out, kind='cells', source=str(path), exact=True)
    (out / 'done').touch(); print(f'\n  {key}: wrote {out}')

def build_effects(key, path, chunk=1024):
    out = PB / key; out.mkdir(parents=True, exist_ok=True)
    with h5py.File(path, 'r') as f:
        obs, var = read_elem(f['obs']), read_elem(f['var'])
        names = obs.index.astype(str).to_numpy(); src = _gene_map(var.index.astype(str))
        measured = np.zeros(len(GENES), bool); ok = src >= 0
        measured[ok] = var['measured'].to_numpy(bool)[src[ok]] if 'measured' in var else True
        ctrl = _aligned(var['ctrl_cpm'].to_numpy(float)[None, :], src)[0]; ctrl[~measured] = np.nan
        keep = np.flatnonzero(names != CONTROL)
        mean = np.lib.format.open_memmap(out / 'mean_cpm.npy', 'w+', np.float32, (len(keep) + 1, len(GENES)))
        mean[0] = ctrl; se_sum = np.zeros(len(GENES)); se_n = np.zeros(len(GENES)); se_med = np.zeros(len(keep))
        X, S = f['X'], f['layers/se']
        for b in range(0, len(keep), chunk):
            rows = keep[b:b + chunk]
            lfc = _aligned(X[rows[0]:rows[-1] + 1][rows - rows[0]], src)
            se = _aligned(S[rows[0]:rows[-1] + 1][rows - rows[0]], src)
            mean[1 + b:1 + b + len(rows)] = np.maximum((ctrl + PC) * np.exp2(lfc) - PC, 0)
            good = np.isfinite(se); se_sum += np.where(good, se, 0).sum(0); se_n += good.sum(0)
            se_med[b:b + len(rows)] = np.nanmedian(se, axis=1)
        mean.flush()
    np.save(out / 'targets.npy', np.r_[[CONTROL], names[keep]].astype(str))
    np.save(out / 'n_cells.npy', np.r_[[np.nan], obs['n_cells'].to_numpy(float)[keep]])
    np.save(out / 'n_reps.npy', np.r_[[np.nan], obs['n_reps'].to_numpy(float)[keep]])
    np.save(out / 'mean_se.npy', se_sum / np.maximum(se_n, 1)); np.save(out / 'target_median_se.npy', se_med)
    np.save(out / 'measured.npy', measured)
    _save_meta(out, kind='effects', source=str(path), exact=False,
               note='mean_cpm reconstructed as (ctrl_cpm+0.5)*2^LFC-0.5; no cells, so no detection or per-cell variance')
    (out / 'done').touch(); print(f'  {key}: wrote {out}')

def ensure_pseudobulk(key):
    path, kind, _ = ALL[key]
    if (PB / key / 'done').exists(): return PB / key
    t0 = time.time(); print(f'building {key} ({kind}) ...')
    {'sums': build_sums, 'cells': build_cells, 'effects': build_effects}[kind](key, path)
    print(f'  {key}: {time.time() - t0:.0f} s'); return PB / key

def load_pb(key):
    d = ensure_pseudobulk(key)
    pb = {'key': key, 'kind': ALL[key][1], 'dir': d}
    for f in d.glob('*.npy'):
        pb[f.stem] = np.load(f, mmap_mode='r', allow_pickle=False)
    pb['meta'] = json.loads((d / 'meta.json').read_text())
    return pb

In [ ]:
# Builds anything not cached yet. H1 (29 GB, 414,694 cells) dominates the first run.
for key in ALL:
    ensure_pseudobulk(key)
PBS = {key: load_pb(key) for key in ALL}

### 2a. Size and depth of each context

* **Cells per knockdown** sets how noisy each knockdown's pseudobulk is (standard error shrinks like $1/\sqrt{n}$).
* **UMIs per cell** (library size) sets how many genes are measurable at all: at 20,000 UMIs, 5 CPM is 0.1 counts per cell.
* **Genes above 5 CPM in controls** is exactly the evaluator's DE gene set for that context.

In [ ]:
def context_table():
    rows = []
    for key, pb in PBS.items():
        n = np.asarray(pb['n_cells'])[1:]; ctrl = np.asarray(pb['mean_cpm'][0]); meas = np.asarray(pb['measured'])
        rows.append(dict(context=key, kind=pb['kind'], knockdowns=len(n),
                         control_cells=float(np.asarray(pb['n_cells'])[0]),
                         cells_per_kd_median=np.nanmedian(n) if len(n) else np.nan,
                         cells_per_kd_iqr=f'{np.nanpercentile(n,25):.0f}-{np.nanpercentile(n,75):.0f}' if len(n) and np.isfinite(n).any() else '',
                         ctrl_umi_per_cell=float(np.asarray(pb['umi_per_cell'])[0]) if 'umi_per_cell' in pb else np.nan,
                         genes_measured=int(meas.sum()),
                         genes_ctrl_gt5cpm=int(np.nansum(ctrl > 5)),
                         panel_targets_present=int(np.isin(PANEL, np.asarray(pb['targets'])).sum())))
    return pd.DataFrame(rows).set_index('context')
SUMMARY = context_table(); SUMMARY

### 2b. Noise and spread, context by context

Four panels per context:

1. **Cells per knockdown** (log scale).
2. **Depth**: UMIs per cell, per cell for cell-level contexts and per knockdown average for Orion.
3. **Control mean–variance**: each dot is a gene. The dashed line is pure Poisson counting noise at the median depth, $\operatorname{Var}(y)\approx \bar y\cdot 10^6/L$. Genes above it are **overdispersed**: biological cell-to-cell variability on top of counting noise.
4. **Signal vs noise across knockdowns** (knockdowns with at least 20 cells), binned by control expression. The blue line is the median **across-knockdown spread**, the SD over knockdowns of $\log_2\frac{\bar y_{pg}+0.5}{\bar y_{0g}+0.5}$. The orange line is the median **expected sampling SE** of that same quantity for a typical knockdown. Where blue ≈ orange, differences between knockdowns are mostly noise. Where blue ≫ orange, there is real knockdown-specific signal.

In [ ]:
def kd_rows(pb):
    """Knockdown rows with at least MIN_CELLS cells (rows of unknown size are kept)."""
    n = np.asarray(pb['n_cells'], float)
    return np.flatnonzero((np.arange(len(n)) > 0) & ~(n < MIN_CELLS))

def lfc_stats(pb, chunk=1024):
    """Per gene: SD of log2FC across knockdowns. Per knockdown: #genes |log2FC|>1 among genes > 5 CPM in control.
    Only knockdowns with >= MIN_CELLS cells; `rows` says which pseudobulk rows the per-knockdown arrays refer to."""
    m = pb['mean_cpm']; ctrl = np.asarray(m[0], float); gated = np.nan_to_num(ctrl) > 5
    s1 = np.zeros(len(GENES)); s2 = np.zeros(len(GENES)); cnt = np.zeros(len(GENES))
    n_de, l2 = [], []; rows = kd_rows(pb)
    for b in range(0, len(rows), chunk):
        x = np.log2((np.asarray(m[rows[b:b + chunk]], float) + PC) / (ctrl + PC))
        ok = np.isfinite(x); xz = np.where(ok, x, 0)
        s1 += xz.sum(0); s2 += (xz ** 2).sum(0); cnt += ok.sum(0)
        n_de.append((np.abs(np.where(gated, xz, 0)) > 1).sum(1)); l2.append(np.sqrt((np.where(gated, xz, 0) ** 2).sum(1)))
    sd = np.sqrt(np.maximum(s2 / np.maximum(cnt, 1) - (s1 / np.maximum(cnt, 1)) ** 2, 0)); sd[cnt < 3] = np.nan
    return dict(sd=sd, rows=rows, n_de=np.concatenate(n_de) if n_de else np.array([]), l2=np.concatenate(l2) if l2 else np.array([]), gated=gated)

def expected_se(pb):
    ctrl = np.asarray(pb['mean_cpm'][0], float)
    if pb['kind'] == 'effects':
        return np.asarray(pb['mean_se'], float)
    n_kd = np.nanmedian(np.asarray(pb['n_cells'], float)[kd_rows(pb)]) if pb['n_cells'].shape[0] > 1 else np.nan
    var = np.asarray(pb['ctrl_var_cpm'], float)
    return np.sqrt(var / n_kd) / (ctrl + PC) / np.log(2)

STATS = {}
def plot_context(key):
    pb = PBS[key]; ctrl = np.asarray(pb['mean_cpm'][0], float)
    fig, ax = plt.subplots(1, 4, figsize=(19, 3.6)); fig.suptitle(f'{key}  ({pb["kind"]})', x=0.01, ha='left', fontweight='bold')
    n = np.asarray(pb['n_cells'], float)[1:]
    if len(n) and np.isfinite(n).any():
        ax[0].hist(n[np.isfinite(n)], bins=np.logspace(0, np.log10(np.nanmax(n) + 1), 40)); ax[0].set_xscale('log')
        ax[0].axvline(MIN_CELLS, color='r', ls=':', lw=1); ax[0].set_title(f'cells per knockdown (median {np.nanmedian(n):.0f}; {np.mean(n < MIN_CELLS):.1%} < {MIN_CELLS})')
    else:
        ax[0].text(.5, .5, 'controls only', ha='center', transform=ax[0].transAxes); ax[0].set_title('cells per knockdown')
    if 'cell_library' in pb:
        lib = np.asarray(pb['cell_library']); ax[1].hist(lib, bins=60); ax[1].set_title(f'UMIs per cell (median {np.median(lib):,.0f})')
    elif 'umi_per_cell' in pb:
        u = np.asarray(pb['umi_per_cell'])[1:]; ax[1].hist(u, bins=60); ax[1].set_title(f'mean UMIs/cell per knockdown (median {np.median(u):,.0f})')
    else:
        ax[1].text(.5, .5, 'no cell-level data released', ha='center', transform=ax[1].transAxes); ax[1].set_title('depth')
    if 'ctrl_var_cpm' in pb:
        var = np.asarray(pb['ctrl_var_cpm'], float); ok = (ctrl > 0) & (var > 0)
        ax[2].loglog(ctrl[ok], var[ok], '.', ms=1.5, alpha=.3)
        depth = np.median(np.asarray(pb['cell_library'])) if 'cell_library' in pb else float(np.asarray(pb['umi_per_cell'])[0])
        xs = np.logspace(np.log10(ctrl[ok].min()), np.log10(ctrl[ok].max()), 50)
        ax[2].loglog(xs, xs * 1e6 / depth, 'k--', lw=1, label='Poisson'); ax[2].legend(loc='upper left')
        ax[2].set_xlabel('control mean CPM'); ax[2].set_ylabel('variance of per-cell CPM'); ax[2].set_title('control mean-variance')
    else:
        ok = ctrl > 0; ax[2].hist(np.log10(ctrl[ok]), bins=60); ax[2].set_xlabel('log10 control CPM'); ax[2].set_title('control expression (no per-cell variance)')
    if pb['mean_cpm'].shape[0] > 3:
        st = lfc_stats(pb); se = expected_se(pb); STATS[key] = dict(st, se=se)
        bins = np.arange(-1, 4.01, .25); x = np.log10(ctrl + 1e-3); mid = (bins[1:] + bins[:-1]) / 2
        sd_b = [np.nanmedian(st['sd'][(x >= lo) & (x < hi)]) for lo, hi in zip(bins[:-1], bins[1:])]
        se_b = [np.nanmedian(se[(x >= lo) & (x < hi)]) for lo, hi in zip(bins[:-1], bins[1:])]
        ax[3].plot(mid, sd_b, label='spread across knockdowns'); ax[3].plot(mid, se_b, label='expected sampling SE')
        ax[3].axvline(np.log10(5), color='grey', ls=':', lw=1); ax[3].text(np.log10(5), ax[3].get_ylim()[1]*.9, ' 5 CPM gate', color='grey', fontsize=8)
        ax[3].set_xlabel('log10 control CPM'); ax[3].set_ylabel('log2 FC'); ax[3].set_title('signal vs noise'); ax[3].legend()
    else:
        ax[3].text(.5, .5, 'controls only:\nno knockdowns to compare', ha='center', va='center', transform=ax[3].transAxes); ax[3].set_title('signal vs noise')
    plt.tight_layout(); plt.show()

for key in ALL:
    plot_context(key)

### 2c. How strong are the knockdowns?

For each knockdown with at least 20 cells: the number of genes with $|\log_2 FC| > 1$ among genes above 5 CPM in that context's controls, which is the gene set the evaluator tests. The challenge panel's 300 targets are shown separately where a context measured them. Most knockdowns move very few genes; the response is concentrated in a minority of strong perturbations.

In [ ]:
rows = []
fig, ax = plt.subplots(1, len(STATS), figsize=(4 * len(STATS), 3.2), squeeze=False)
for i, (key, st) in enumerate(STATS.items()):
    t = np.asarray(PBS[key]['targets'])[st['rows']]; nde = st['n_de']; in_panel = np.isin(t, PANEL)
    ax[0, i].hist(np.log10(nde + 1), bins=40, alpha=.6, label='all knockdowns', density=True)
    if in_panel.any(): ax[0, i].hist(np.log10(nde[in_panel] + 1), bins=20, alpha=.6, label='challenge panel', density=True)
    ax[0, i].set_title(key); ax[0, i].set_xlabel('log10(1 + #genes |log2FC|>1)'); ax[0, i].legend(fontsize=7)
    rows.append(dict(context=key, knockdowns=len(nde), median_n_de=np.median(nde), p90_n_de=np.percentile(nde, 90),
                     frac_with_0=np.mean(nde == 0), panel_median_n_de=np.median(nde[in_panel]) if in_panel.any() else np.nan))
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index('context'))
for key, st in STATS.items():
    t = np.asarray(PBS[key]['targets'])[st['rows']]; top = np.argsort(-st['n_de'])[:10]
    print(f'{key:11s} strongest:', ', '.join(f'{t[j]}({st["n_de"][j]})' for j in top))

### 2d. How similar are the control transcriptomes?

Pearson correlation of $\log(1+\text{mean CPM})$ over genes measured everywhere and expressed somewhere. This is close to what the transfer model uses to weight sources for each challenge line. Platform matters as well as biology: H1 used 10x Flex probes, Orion and the challenge lines 3′ capture.

In [ ]:
keys = list(ALL)
ctrl = np.vstack([np.asarray(PBS[k]['mean_cpm'][0], float) for k in keys])
ok = np.isfinite(ctrl).all(0) & (np.nan_to_num(ctrl).max(0) > 1)
C = np.corrcoef(np.log1p(ctrl[:, ok]))
fig, ax = plt.subplots(figsize=(6.2, 5.2)); im = ax.imshow(C, vmin=np.floor(C.min() * 10) / 10, vmax=1, cmap='viridis')
ax.set_xticks(range(len(keys)), keys, rotation=45, ha='right'); ax.set_yticks(range(len(keys)), keys)
for i in range(len(keys)):
    for j in range(len(keys)): ax.text(j, i, f'{C[i, j]:.2f}', ha='center', va='center', fontsize=7, color='w' if C[i, j] < C.mean() else 'k')
plt.colorbar(im, fraction=.046); ax.set_title(f'control profile correlation ({ok.sum():,} genes)'); ax.grid(False); plt.tight_layout(); plt.show()

### 2e. Look up one gene or one knockdown

`gene_profile('GAPDH')` shows a gene's control expression and detection in every context. `knockdown_profile('MED13')` shows a knockdown's top responding genes in every context that measured it, using the pseudobulk rather than the pipeline's effect tables.

In [ ]:
def gene_profile(gene):
    j = GENE_POS[gene]; rows = []
    for key, pb in PBS.items():
        rows.append(dict(context=key, ctrl_mean_cpm=float(pb['mean_cpm'][0, j]), ctrl_pooled_cpm=float(pb['pooled_cpm'][0, j]) if 'pooled_cpm' in pb else np.nan,
                         ctrl_detection=float(pb['detection'][0, j]) if 'detection' in pb else np.nan,
                         knocked_down_here=gene in set(np.asarray(pb['targets'])), in_challenge_panel=gene in set(PANEL)))
    return pd.DataFrame(rows).set_index('context')

def knockdown_profile(target, n=8):
    out = {}
    for key, pb in PBS.items():
        t = np.asarray(pb['targets']); hit = np.flatnonzero(t == target)
        if not len(hit): continue
        ctrl = np.asarray(pb['mean_cpm'][0], float); x = np.log2((np.asarray(pb['mean_cpm'][hit[0]], float) + PC) / (ctrl + PC))
        x[~(np.nan_to_num(ctrl) > 5)] = np.nan; j = GENE_POS.get(target)
        own = f'{x[j]:+.2f}' if j is not None and np.isfinite(x[j]) else 'n/a'
        order = np.argsort(np.nan_to_num(x, nan=0)); n_cells = np.asarray(pb['n_cells'])[hit[0]]
        out[f'{key} (n={n_cells:.0f}, own {own})'] = [f'{GENES[i]} {x[i]:+.2f}' for i in order[:n]] + [f'{GENES[i]} {x[i]:+.2f}' for i in order[::-1][:n]]
    if not out: print(f'{target} is not knocked down in any context'); return None
    return pd.DataFrame(out, index=[f'down {i+1}' for i in range(n)] + [f'up {i+1}' for i in range(n)])

display(gene_profile(PANEL[0])); display(knockdown_profile(PANEL[0]))

## 3. The pipeline's effect tables

Built by the effects job into `~/vcc/lib/review_v4/effects/<name>/`. Each folder holds a knockdowns × genes matrix of log2 fold changes (`lfc.npy`) and standard errors (`se.npy`), stored as float16, plus the guide QC log and the control profile.

Labels use per-cell CPM with a **0.5 CPM pseudocount** and an SE floor of one Poisson count, so unscored low-count genes cannot produce huge, falsely precise labels. The pipeline shrinks each label toward 0 by $\frac{\sigma_p^2}{\sigma_p^2+SE^2}$, where $\sigma_p^2$ is the context's prior variance (shown below).

This section reads the tables through the pipeline's own `Effects` class, so gene alignment matches exactly what the model sees.

In [ ]:
from vcc_common import Effects, load_profile, shrink
EFF_DIRS = {p.parent.name: p.parent for p in sorted((WORK / 'effects').glob('*/meta.json'))}
print('effect tables found:', list(EFF_DIRS) or 'none yet: run the effects job in vcc_2026.ipynb first')
EFF = {name: Effects(d, GENES) for name, d in EFF_DIRS.items()}
EFF_PROF = {name: load_profile(d / 'control_profile.npz', GENES) for name, d in EFF_DIRS.items()}

def effect_overview():
    rows = []
    for name, e in EFF.items():
        qc = pd.read_csv(EFF_DIRS[name] / 'guide_qc.csv') if (EFF_DIRS[name] / 'guide_qc.csv').exists() else pd.DataFrame()
        m = e.meta
        rows.append(dict(table=name, knockdowns=len(e.targets),
                         guide_rows_checked=len(qc), guide_rows_failed=int((~qc.passed.astype(bool)).sum()) if len(qc) else np.nan,
                         estimator=m.get('label_estimator', m.get('lfc', ''))[:40], pseudocount=m.get('label_pseudocount', ''),
                         se_kind=str(m.get('se_kind', ''))[:45], prior_variance=e.prior_variance,
                         genes_gt5cpm=int(np.nansum(EFF_PROF[name]['mean_cpm'] > 5)),
                         panel_covered=int(np.isin(PANEL, e.targets).sum())))
    return pd.DataFrame(rows).set_index('table')
t0 = time.time(); OVERVIEW = effect_overview(); print(f'{time.time()-t0:.0f} s (prior variance reads every row once)'); OVERVIEW

### 3a. Guide QC, label and SE distributions

* **On-target LFC**: how far each guide row knocked down its own target. Rows at or above −0.1 were dropped before choosing a guide.
* **LFC and SE histograms**: 300 random knockdowns, restricted to genes above 5 CPM in that context's controls.
* **Shrinkage**: raw versus shrunk LFC on the same sample. Points pulled hard toward 0 had large SEs relative to the context's prior.

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(len(EFF), 4, figsize=(19, 3.1 * len(EFF)), squeeze=False)
for i, (name, e) in enumerate(EFF.items()):
    qc_path = EFF_DIRS[name] / 'guide_qc.csv'
    if qc_path.exists():
        qc = pd.read_csv(qc_path); v = qc.ontarget_lfc.dropna().clip(-6, 3)
        ax[i, 0].hist(v, bins=60); ax[i, 0].axvline(-0.1, color='r', ls='--', lw=1)
        ax[i, 0].set_title(f'{name}: on-target LFC ({(qc.ontarget_lfc >= -0.1).mean():.1%} rows failed)', fontsize=9)
    gated = np.flatnonzero(np.nan_to_num(EFF_PROF[name]['mean_cpm']) > 5)
    pick = rng.choice(len(e.targets), min(300, len(e.targets)), replace=False)
    L, S = [], []
    for r in pick:
        l, s = e.get(e.targets[r], gated); L.append(l); S.append(s)
    L, S = np.concatenate(L), np.concatenate(S); ok = np.isfinite(L) & np.isfinite(S)
    ax[i, 1].hist(np.clip(L[ok], -3, 3), bins=120, log=True); ax[i, 1].set_title(f'LFC, genes >5 CPM (sd {np.std(L[ok]):.3f})', fontsize=9)
    ax[i, 2].hist(np.clip(S[ok], 0, 2), bins=100, log=True); ax[i, 2].set_title(f'SE (median {np.median(S[ok]):.3f})', fontsize=9)
    sh = shrink(L[ok], S[ok], prior_variance=e.prior_variance); sub = rng.choice(ok.sum(), min(20000, ok.sum()), replace=False)
    ax[i, 3].plot(L[ok][sub], sh[sub], ',', alpha=.4); ax[i, 3].plot([-3, 3], [-3, 3], 'k:', lw=1)
    ax[i, 3].set_xlim(-3, 3); ax[i, 3].set_ylim(-3, 3); ax[i, 3].set_xlabel('raw'); ax[i, 3].set_ylabel('shrunk')
    ax[i, 3].set_title(f'shrinkage (prior var {e.prior_variance:.3f})', fontsize=9)
plt.tight_layout(); plt.show()

### 3b. Do knockdown effects transfer between backgrounds?

For every pair of tables, take the knockdowns both measured (up to 400, sampled) and correlate their LFC vectors over genes above 5 CPM in **both** controls. The median correlation is a direct, model-free read of how much a knockdown's signature carries over from one cell type to another. It is the ceiling for any "copy the source effect" strategy.

Two versions:

* **All expressed genes.** Correlate over every gene above 5 CPM in both controls. Most of those genes don't respond, so this dilutes the signal and reads low.
* **Responding genes.** Correlate only over genes with $|\text{LFC}|>0.5$ in either context, at least 10 of them. This asks: where something happens, does it happen the same way? It's closer to what the DE metrics reward.

Each is also shown for **strong knockdowns** only, where both sides move at least 20 expressed genes by more than 2-fold.

In [ ]:
def lfc_matrix(e, targets, gene_idx):
    return np.vstack([e.get(t, gene_idx)[0] for t in targets]).astype(float)

names = list(EFF); shared_n = pd.DataFrame(index=names, columns=names, dtype=float)
TABLES = {k: shared_n.copy() for k in ('all genes', 'all genes, strong knockdowns', 'responding genes', 'responding genes, strong knockdowns')}
for a_ in names:
    for b_ in names:
        if a_ >= b_: continue
        shared = sorted(set(EFF[a_].targets) & set(EFF[b_].targets)); shared_n.loc[a_, b_] = shared_n.loc[b_, a_] = len(shared)
        if len(shared) < 5: continue
        shared = list(rng.choice(shared, min(400, len(shared)), replace=False))
        g = np.flatnonzero((np.nan_to_num(EFF_PROF[a_]['mean_cpm']) > 5) & (np.nan_to_num(EFF_PROF[b_]['mean_cpm']) > 5))
        A, B = lfc_matrix(EFF[a_], shared, g), lfc_matrix(EFF[b_], shared, g)
        res = {k: [] for k in TABLES}
        def corr(u, v):
            return np.corrcoef(u, v)[0, 1] if len(u) >= 10 and np.std(u) > 0 and np.std(v) > 0 else np.nan
        for x, y in zip(A, B):
            ok = np.isfinite(x) & np.isfinite(y)
            if ok.sum() < 50: continue
            xs, ys = x[ok], y[ok]; is_strong = (np.abs(xs) > 1).sum() >= 20 and (np.abs(ys) > 1).sum() >= 20
            resp = (np.abs(xs) > 0.5) | (np.abs(ys) > 0.5)
            for k, v in (('all genes', corr(xs, ys)), ('responding genes', corr(xs[resp], ys[resp]))):
                if np.isfinite(v):
                    res[k].append(v)
                    if is_strong: res[k + ', strong knockdowns'].append(v)
        for k, v in res.items():
            TABLES[k].loc[a_, b_] = TABLES[k].loc[b_, a_] = np.median(v) if v else np.nan
display(Markdown('**shared knockdowns**')); display(shared_n.fillna(0).astype(int))
for k, tab in TABLES.items():
    display(Markdown(f'**median per-knockdown LFC correlation: {k}**')); display(tab.round(3))

### 3c. Look up a knockdown in the effect tables

`effect_lookup('MED13')` lists, for each table that measured it, the on-target LFC and the strongest shrunk responses among genes above 5 CPM, with each label's SE.

In [ ]:
def effect_lookup(target, n=8):
    out = {}
    for name, e in EFF.items():
        if target not in e.pos: continue
        gated = np.flatnonzero(np.nan_to_num(EFF_PROF[name]['mean_cpm']) > 5)
        l, s = e.get(target, gated); sh = shrink(l, s, prior_variance=e.prior_variance)
        j = GENE_POS.get(target); own = e.get(target, [j])[0][0] if j is not None else np.nan
        order = np.argsort(np.nan_to_num(sh))
        pick = list(order[:n]) + list(order[::-1][:n])
        out[f'{name} (own {own:+.2f})'] = [f'{GENES[gated[i]]} {sh[i]:+.2f} (se {s[i]:.2f})' for i in pick]
    if not out: print(f'{target} is in no effect table'); return None
    return pd.DataFrame(out, index=[f'down {i+1}' for i in range(n)] + [f'up {i+1}' for i in range(n)])

effect_lookup(PANEL[0])

## 4. The six 2026 metrics, as equations

Transcribed from the installed evaluator, `cell-eval2` 0.18.0 with the `vcc2026` preset: `catalog.py`, `metrics/discrimination.py`, `metrics/delta.py`, `metrics/de.py`, `metrics/direction.py`, `de_compute.py`, `scoring.py`. Where the code and its docs disagree, the code wins.

### 4.0 Notation and the two expression spaces

* Perturbations $p\in\mathcal P$ (the 300 panel targets of one line), with target gene $g_p$. The real control population is $0$. Genes $g\in G$, $|G| = 18{,}533$.
* Cell $c$ has raw counts $x_{cg}$ and library size $L_c=\sum_g x_{cg}$. A side $s\in\{\text{pred},\text{real}\}$ is your submitted cells or the held-out real cells.
* **The control is always the real one.** Under `control_source = real`, your perturbed cells are compared against Arc's held-out control cells, never against cells you submit. Submissions contain no controls.

**DE space**, used by metrics 3–6. Per-cell CPM, then the arithmetic mean over the group:
$$y_{cg}=\frac{10^6\,x_{cg}}{L_c},\qquad \bar y^{\,s}_{pg}=\frac1{n_p}\sum_{c\in p}y_{cg}.$$

**Expression-comparator space**, used by metrics 1–2. Pool the group first, then log-normalize to $T=5\times10^4$ (`bulk_target_sum`):
$$\mu^{s}_{pg}=\log\!\Big(1+T\,\frac{\sum_{c\in p}x_{cg}}{\sum_{c\in p}L_c}\Big).$$

These differ when library size co-varies with composition. Metrics 1–2 see pooled composition, while metrics 3–6 see the average per-cell composition.

### 4.1 The DE tables, computed once per side and shared by metrics 3–6

For each perturbation $p$ and side $s$, against the real control:

$$\ell^{s}_{pg}=\log_2\frac{\bar y^{\,s}_{pg}+\varepsilon}{\bar y^{\,\text{real}}_{0g}+\varepsilon},\qquad\varepsilon=10^{-9}.$$

* **Test:** two-sided Wilcoxon rank-sum (Mann–Whitney U) of $\{y_{cg}:c\in p\}$ against $\{y_{cg}:c\in 0\}$, with the pdex backend.
* **Gene gate, reference-only:** $H=\{g:\bar y^{\,\text{real}}_{0g}>5\}$, decided by the real controls alone and identical for both sides and every perturbation. Genes that are silent in controls can never be scored as DE, even if strongly induced.
* **Multiple testing:** $q^{s}_{pg}$ = Benjamini–Hochberg adjusted p-values over $g\in H$, recomputed **separately for each perturbation and each side** after the gate.
* **Significant sets, own target excluded:** $\alpha = 0.05$, with
$$R_p=\{g\in H\setminus\{g_p\}: q^{\text{real}}_{pg}<\alpha\},\qquad P_p=\{g\in H\setminus\{g_p\}: q^{\text{pred}}_{pg}<\alpha\}.$$
* **Direction is defined** when an LFC is finite or infinite but nonzero and not NaN: $\operatorname{def}(\ell)\iff \ell\notin\{0,\text{NaN},\text{null}\}$.

---

### Metric 1: `pds_cosine` (perturbation discrimination; higher is better; random ≈ 0.5, perfect = 1)

Effects relative to the real control, with **every panel target gene removed** from the feature space ($G' = G\setminus\{g_q:q\in\mathcal P\}$):
$$e^{s}_{p}=\big(\mu^{s}_{pg}-\mu^{\text{real}}_{0g}\big)_{g\in G'}.$$
Cosine distance from each predicted effect to every real effect:
$$d_{ij}=1-\frac{\langle e^{\text{pred}}_i,\,e^{\text{real}}_j\rangle}{\lVert e^{\text{pred}}_i\rVert\,\lVert e^{\text{real}}_j\rVert}.$$
Mid-rank of the true match among the $n=|\mathcal P|$ real perturbations, including itself:
$$\rho_i=\#\{j:d_{ij}<d_{ii}\}+\tfrac12\big(\#\{j:d_{ij}=d_{ii}\}-1\big),\qquad \text{PDS}_i=1-\frac{\rho_i}{n-1},\qquad \text{pds\_cosine}=\frac1n\sum_i\text{PDS}_i.$$
Scale-free: only the direction of each effect vector matters. It rewards **distinct** per-perturbation signatures and penalizes anything shared by all perturbations.

---

### Metric 2: `expr_mse_unbiased_capped_norm` (expression error; lower is better; 0 = perfect, 1 = "predict the control")

Per perturbation, over its scored genes $G_p=G\setminus\{g_p\}$. The sampling-noise term $C^{s}_p$ is a delete-one-cell jackknife estimate of the noise in $\mu^{s}_p$:
$$C^{s}_p=\frac{n_p-1}{n_p}\sum_{c\in p}\big\lVert\mu^{s,(-c)}_p-\bar\mu^{s,(\cdot)}_p\big\rVert^2.$$
Here $\mu^{(-c)}$ is the pooled profile recomputed without cell $c$, and $\bar\mu^{(\cdot)}$ is the mean of those $n_p$ profiles.

Debiased, capped squared error, and the debiased real effect size:
$$N_p=\frac{\lVert\mu^{\text{pred}}_p-\mu^{\text{real}}_p\rVert^2_{G_p}\;-\;r\cdot\min\!\big(C^{\text{pred}}_p,\;k\,C^{\text{real}}_p\big)\;-\;C^{\text{real}}_p}{|G_p|},\qquad k=1,$$
$$D_p=\frac{\lVert\mu^{\text{real}}_p-\mu^{\text{real}}_0\rVert^2_{G_p}\;-\;C^{\text{real}}_p\;-\;C^{\text{real}}_0}{|G_p|}.$$
The budget factor $r$ stops a submission claiming more noise correction than its own across-perturbation spread can justify, with weights $w_p=1/|G_p|$:
$$r=\min\!\Big(1,\;\frac{B}{\sum_q w_q\min(C^{\text{pred}}_q,\,kC^{\text{real}}_q)}\Big),\qquad B=\sum_{g}\sum_{q:\,g\neq g_q}w_q\big(\mu^{\text{pred}}_{qg}-\bar\mu^{\text{pred}}_{\cdot g}\big)^2.$$
The aggregate is a **ratio of sums**, not a mean of ratios:
$$\text{expr\_mse\_unbiased\_capped\_norm}=\frac{\sum_p N_p}{\sum_p D_p}.$$
Submitting the control profile for every perturbation gives about 1, and perfect prediction gives about 0. The noise you put into your 400 cells is forgiven only up to the real data's own noise level.

---

### Metric 3: `de_wilcoxon_lfc_nmae` (LFC magnitude error; lower is better; 0 = perfect, 1 = "predict no change")

Over the real-significant genes with finite real LFC. A missing or non-finite predicted LFC counts as $0$.
$$\text{nMAE}_p=\frac{\sum_{g\in R_p}\big|\ell^{\text{pred}}_{pg}-\ell^{\text{real}}_{pg}\big|}{\sum_{g\in R_p}\big|\ell^{\text{real}}_{pg}\big|}.$$
A perturbation is **omitted**, identically for everyone, if $|R_p|<10$ or the denominator is 0. The metric is the mean over the remaining perturbations.

This is the only DE metric that cares about the **size** of the fold changes, not just which genes or which sign.

---

### Metric 4: `de_wilcoxon_sig_jaccard` (DE-set overlap; higher is better)
$$J_p=\frac{|R_p\cap P_p|}{|R_p\cup P_p|},\qquad J_p=1\text{ if }R_p=P_p=\varnothing,\qquad \text{sig\_jaccard}=\frac1n\sum_pJ_p.$$
Penalizes missed real DE genes **and** spurious predicted ones. Your $P_p$ depends on your fold changes **and on your noise and cell count**, because $q^{\text{pred}}$ comes from a Wilcoxon test on your 400 cells against the real controls.

---

### Metric 5: `de_wilcoxon_direction_fidelity_yield_raw` (direction agreement; higher is better)

Define the real budget $N_p=|R_p|$, and your scored set as your significant genes whose real direction is defined, $S_p=\{g\in P_p:\operatorname{def}(\ell^{\text{real}}_{pg})\}$, with $n^{\text{pred}}_p=|S_p|$. Count the sign matches:
$$k_p=\#\{g\in S_p:\operatorname{def}(\ell^{\text{pred}}_{pg})\ \wedge\ \operatorname{sign}\ell^{\text{pred}}_{pg}=\operatorname{sign}\ell^{\text{real}}_{pg}\}.$$
Then
$$\text{FY}_p=\frac{k_p}{\max\big(n^{\text{pred}}_p,\,N_p\big)}.$$
The value is undefined and omitted when both are 0, and the metric is the mean over the rest.

Calling more genes than the real budget lowers it through $n^{\text{pred}}_p$. Calling fewer caps it at $k_p/N_p$. The best is to call about $N_p$ genes, all with the right sign.

---

### Metric 6: `de_wilcoxon_direction_reach_raw` (how deep your ranking stays right; higher is better)

Take the real-significant genes $R_p$ and order them by **your** evidence. Your significant genes come first, then ascending $q^{\text{pred}}$, then ascending $p^{\text{pred}}$, then descending $|\ell^{\text{pred}}|$, then gene name. Walking down that list, counting only adjudicable genes (those with $\operatorname{def}(\ell^{\text{real}})$):
$$\text{purity}_p(k)=\frac{\#\{\text{sign matches among the first }k\}}{k},\qquad k^*_p=\max\{k:\text{purity}_p(k)\ge 0.9\}\ (0\text{ if none}),$$
$$\text{reach}_p=\frac{k^*_p}{N_p},\quad N_p=|R_p|>0;\qquad \text{direction\_reach\_raw}=\text{mean}_p\,\text{reach}_p.$$
This rewards putting your **most confident** calls on genes whose direction you have right. A wrong sign at the top of your ranking is expensive.

---

### 4.7 From raw values to the leaderboard score

For each metric $m$ on a line, Arc measures two reference points on the held-out data:

* $b_m$: the **generic-response baseline**. Every perturbation is predicted as the panel's average real response, emitted as cells.
* $r_m$: the **replicate anchor**. Real data split in half and scored against itself, averaged over 5 splits, which is what a perfect biological replicate achieves.

$$s_m=\frac{u_m-b_m}{r_m-b_m}\qquad(0=\text{baseline},\ 1=\text{as good as a replicate; same formula for lower-is-better metrics}).$$

Clamps on $s_m$:

| Metric | Clamp |
|---|---|
| `expr_mse_unbiased_capped_norm` | $[0,1]$ |
| `de_wilcoxon_lfc_nmae` | linear below 0, floored at $-6$ |
| the other four | no floor or ceiling, so they can go negative or above 1 |

The line score is $\text{avg\_score}=\frac16\sum_m s_m$.

### 4.8 What moves each metric

| Metric | Space | Sensitive to | Not sensitive to |
|---|---|---|---|
| pds_cosine | pooled log-normalized | direction of each effect vector; distinctness between perturbations | overall scale; panel target genes |
| expr_mse…norm | pooled log-normalized | per-gene effect size (shrinking to 0 → 1.0); noise beyond the real data's | own target gene |
| lfc_nmae | per-cell CPM, real-significant genes | magnitude of LFC on genes that really change | genes that don't really change |
| sig_jaccard | DE calls | which genes reach q<0.05 in your 400 cells: magnitude, noise, cell count | sign |
| direction_fidelity_yield_raw | DE calls | sign on your calls; calling about as many as truth | magnitude beyond reaching significance |
| direction_reach_raw | DE ranking | sign of your top-ranked genes | genes outside the real-significant set |

Two practical consequences for our generator:

* The 5 CPM gate is decided by real controls, so predicted induction of control-silent genes affects only metrics 1–2.
* The DE metrics are computed on your actual cells, so the count-noise model (overdispersion, the up_w blend, library size) shifts $P_p$ and $q^{\text{pred}}$ directly, not just the mean.

## 5. What we measure about control cells and every cell line

The pipeline turns each population of control cells into a **profile**, a `.npz` file. The challenge lines' profiles are at `lib/review_v4/profiles/{A,B,C}.npz`. Each training source's profile is `control_profile.npz` inside its effect folder. The transfer model compares a challenge line's profile to each source's, and that comparison is the only information it has about a new line.

### 5a. The fields in a profile

| Field | Per | Meaning | Used for |
|---|---|---|---|
| `mean_cpm` | gene | mean of per-cell CPM, the evaluator's DE convention | the 5 CPM scoring gate (`scored`) |
| `pooled_cpm` / `bulk_cpm` | gene | ratio-of-sums CPM; comparable between cell-level and pseudobulk sources | **the transfer baseline** |
| `log_mean` | gene | $\log(1+\text{pooled CPM})$ | baseline features and similarity |
| `detection` | gene | fraction of control cells with ≥1 count | detection gate on the knocked-down gene |
| `var_cpm`, `se_mean` | gene | spread of per-cell CPM, and SE of the mean | diagnostics |
| `scored` | gene | `mean_cpm > 5` | mirrors the evaluator's DE gene set |
| `measured` | gene | gene present in this source | masking |
| `program_names`, `program_levels` | program | mean $\log(1+\text{pooled CPM})$ over 8 gene programs: S phase, G2/M, stress, interferon, hypoxia/glycolysis, ribosome biogenesis, ribosomal proteins, mitochondrial | line-level context features |
| ⚠️ ribosome program | — | always 0: the 2026 gene list (and H1) contains **no RPL/RPS genes**, so this program is empty everywhere and the pipeline's ribosome features are constant | nothing |
| `n_cells`, `median_library` | profile | control cell count and median UMIs per cell | diagnostics |
| `legacy_baseline` | profile | True when the baseline came from pseudobulk, not cells (T cells) | flags missing per-cell fields |
| `control_obs_names` | cell | which control cells were profiled (challenge lines only) | audit |

In [ ]:
def all_profiles():
    profs = {}
    for k in 'ABC':
        p = WORK / 'profiles' / f'{k}.npz'
        if p.exists(): profs[f'line {k}'] = load_profile(p, GENES)
    for name, d in EFF_DIRS.items():
        profs[name] = load_profile(d / 'control_profile.npz', GENES)
    return profs
PROFS = all_profiles()

rows = []
for name, p in PROFS.items():
    mc = np.asarray(p['mean_cpm'], float); det = np.asarray(p.get('detection', np.full(len(GENES), np.nan)), float)
    rows.append(dict(profile=name, n_cells=int(p.get('n_cells', 0)) or np.nan, median_library=p.get('median_library', np.nan),
                     genes_measured=int(np.isfinite(mc).sum()), genes_scored_gt5cpm=int(np.nansum(mc > 5)),
                     median_detection_scored=np.nanmedian(det[mc > 5]) if np.isfinite(det).any() else np.nan,
                     legacy_baseline=bool(p.get('legacy_baseline', False))))
display(pd.DataFrame(rows).set_index('profile'))
names_ = list(np.asarray(next(iter(PROFS.values()))['program_names']).astype(str))
display(Markdown('**Program levels: mean log(1+pooled CPM) per program**'))
display(pd.DataFrame({n: np.asarray(p['program_levels'], float) for n, p in PROFS.items()}, index=names_).T.round(2))

### 5b. How the model weighs each source for each challenge line

The transfer model computes a **centred similarity**, the Pearson correlation of $\log(1+\text{pooled CPM})$ over shared genes, between a line and each source. It turns those into weights with a softmax at temperature $\tau$:
$$w_{\text{line},s}=\frac{\exp(\text{sim}_{\text{line},s}/\tau)}{\sum_{s'}\exp(\text{sim}_{\text{line},s'}/\tau)}.$$
Within one background, only one table is used, the higher-priority one, so CD4_T contributes its Stim48hr table. $\tau$ is chosen on leave-one-background-out folds. The value used below comes from the most complete `selection.json` available.

In [ ]:
from vcc_transfer_model_v3 import transfer_profile, centred_similarity, DEFAULT_KNOBS, SRC_NAMES, TGT_NAMES
def selected_knobs():
    for p in [WORK / 'models/final/selection.json', *sorted((WORK / 'models').glob('holdout_*/selection.json'))]:
        if p.exists(): return json.loads(p.read_text())['knobs'], str(p.relative_to(WORK))
    return dict(DEFAULT_KNOBS), 'defaults (no selection.json yet)'
KNOBS, KNOB_SRC = selected_knobs(); print('knobs from', KNOB_SRC, '->', KNOBS)

TP = {n: transfer_profile(dict(p), 'bulk', label=n) for n, p in PROFS.items()}
cfg = json.loads((WORK / 'contexts.json').read_text()) if (WORK / 'contexts.json').exists() else {'contexts': []}
chosen = {}
for c in cfg['contexts']:                                   # mirror choose_sources(): one table per background
    best = chosen.get(c['background'])
    if best is None or (c.get('source_priority', 0), c['name']) > (best.get('source_priority', 0), best['name']): chosen[c['background']] = c
src_names = [c['name'] for c in chosen.values()]
lines = [n for n in TP if n.startswith('line')]
sim = pd.DataFrame({s: [centred_similarity(TP[l]['baseline_log'], TP[s]['baseline_log']) for l in lines] for s in TP if not s.startswith('line')}, index=lines)
display(Markdown('**centred similarity, all source tables**')); display(sim.round(4))
w = np.exp(sim[src_names] / KNOBS['temperature']); w = w.div(w.sum(axis=1), axis=0)
display(Markdown(f'**softmax weights at temperature {KNOBS["temperature"]}, the tables actually used**')); display(w.round(3))

### 5c. Expression gates on the 300 challenge targets

A CRISPRi knockdown can only act on a gene the line expresses, so the model multiplies every predicted effect of target $t$ by two gates computed from the line's controls:

$$\text{kgate}_t=\sigma\!\Big(\frac{\log(1+\text{pooled CPM}_t)-c}{s}\Big),\qquad \text{dgate}_t=\sigma\!\Big(\frac{\text{detection}_t-c_d}{s_d}\Big).$$

Each gate is 0 if the target has zero expression or detection, and 1 if the line didn't measure it. Downregulation of a responder gene $g$ is also gated by $g$'s own expression in the line. Targets with a small product $\text{kgate}\times\text{dgate}$ get predicted effects close to zero, which is the model saying the line can't knock down what it doesn't express.

In [ ]:
gate_rows = []
for l in lines:
    p = TP[l]
    for t in PANEL:
        j = GENE_POS.get(t)
        if j is None: gate_rows.append(dict(line=l, target=t, pooled_cpm=np.nan, detection=np.nan, kgate=1.0, dgate=1.0)); continue
        cpm = float(p['baseline_cpm'][j]); det = float(np.asarray(p.get('detection'))[j]) if p.get('detection') is not None else np.nan
        kg = 1.0 if not np.isfinite(cpm) else (0.0 if cpm == 0 else float(expit((np.log1p(cpm) - KNOBS['gate_center']) / KNOBS['gate_scale'])))
        dg = 1.0 if not np.isfinite(det) else (0.0 if det == 0 else float(expit((det - KNOBS['detection_center']) / KNOBS['detection_scale'])))
        gate_rows.append(dict(line=l, target=t, pooled_cpm=cpm, detection=det, kgate=kg, dgate=dg))
GATES = pd.DataFrame(gate_rows); GATES['gate'] = GATES.kgate * GATES.dgate
summary = GATES.groupby('line').agg(targets=('target', 'size'), median_cpm=('pooled_cpm', 'median'),
                                    not_expressed=('pooled_cpm', lambda v: int((v == 0).sum())),
                                    gate_below_0_5=('gate', lambda v: int((v < .5).sum())), gate_below_0_1=('gate', lambda v: int((v < .1).sum())))
display(summary)
fig, ax = plt.subplots(1, len(lines), figsize=(5 * len(lines), 3), squeeze=False)
for i, l in enumerate(lines):
    g = GATES[GATES.line == l]; ax[0, i].hist(np.log10(g.pooled_cpm + 0.1), bins=40)
    ax[0, i].axvline(np.log10(np.expm1(KNOBS['gate_center'])), color='r', ls='--', lw=1, label='kgate = 0.5')
    ax[0, i].set_title(f'{l}: target expression in controls'); ax[0, i].set_xlabel('log10(pooled CPM + 0.1)'); ax[0, i].legend()
plt.tight_layout(); plt.show()
display(Markdown('**Lowest-gated targets per line**'))
display(GATES.sort_values('gate').groupby('line').head(8).sort_values(['line', 'gate']).set_index(['line', 'target']).round(3))

### 5d. The features the transfer model builds per (target, gene, source)

Every prediction for target $t$ and gene $g$ in a line starts from each source's effect of knocking down $t$ on $g$. The model adds features about how that source's context differs from the line. The **source-mean** predictor uses only the gated, shrunk, scaled effect averaged with the similarity weights. **Ridge** adds a correction learned from all the features, and it is used only if it wins at least 3 of 4 leave-one-background-out folds.

In [ ]:
SRC_DOC = {
 'source_lfc': "the source's raw LFC for (t, g), capped at ±8", 'source_se': 'its SE (0 if unknown)', 'se_missing': 'SE unknown or degenerate',
 'source_shrunk_lfc': 'normal-prior shrunk LFC', 'source_baseline': "g's log(1+pooled CPM) in the source controls",
 'baseline_difference': "g's baseline in the line minus in the source", 'source_knockdown_gene_baseline': "t's own baseline in the source",
 'knockdown_gene_baseline_difference': "t's baseline in the line minus in the source", 'source_ontarget_lfc': 'how strongly t was knocked down in the source',
 'ontarget_missing': 'source did not measure t', 'legacy_effect': 'source table has no usable SE', 'context_pearson': 'centred similarity of line vs source controls',
 'agreement': 'fraction of OTHER sources agreeing on the sign', 'legacy_source_baseline': 'source baseline from pseudobulk (T cells)',
 'gated_source_effect': 'shrunk effect x per-source scale x expression gates: what source_mean averages', 'output_expression_gate': "gate on g's expression in the line (applied to down effects)",
 'knockdown_expression_gate': 'kgate for t', 'knockdown_detection_gate': 'dgate for t', 'source_scale': 'one learned scale per source table (platform/LFC compression)'}
for p in names_: SRC_DOC[f'program_difference_{p}'] = f'line minus source level of the {p} program'
TGT_DOC = {'target_baseline': "g's log(1+pooled CPM) in the line", 'target_knockdown_gene_baseline': "t's baseline in the line",
           'knockdown_gene_missing': 'line did not measure t', 'legacy_target_baseline': 'line baseline from pseudobulk'}
for p in names_: TGT_DOC[f'target_program_{p}'] = f'line level of the {p} program'
display(pd.DataFrame({'per source (averaged with similarity weights, plus their spread)': pd.Series(SRC_DOC)}).rename_axis('feature'))
display(pd.DataFrame({'per line (shared by all sources)': pd.Series(TGT_DOC)}).rename_axis('feature'))
assert set(SRC_DOC) == set(SRC_NAMES) and set(TGT_DOC) == set(TGT_NAMES), 'feature list changed in the pipeline: update this table'